In [ ]:
import requests
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd

def download_file(file_url: str, save_as: str) -> None:
    """Download a file from a URL and save it locally."""
    response = requests.get(file_url)
    response.raise_for_status()
    with open(save_as, "wb") as file:
        file.write(response.content)

def extract_column_names(file_path: str, sheet_name: str) -> list:
    """Extract and clean column names from the Excel file."""
    raw_columns = pd.read_excel(
        file_path, 
        sheet_name=sheet_name, 
        skiprows=1, 
        nrows=7, 
        header=None
    ).transpose()
    return [
        ' '.join(name.split()) 
        for name in raw_columns.apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1).iloc[1:].tolist()
    ]

def parse_dates(file_path: str, sheet_name: str) -> pd.DatetimeIndex:
    """Parse and process the dates from the Excel file."""
    date_data = pd.read_excel(
        file_path, 
        sheet_name=sheet_name, 
        skiprows=7, 
        usecols=['Date'], 
        dtype={'Date': str}
    ).dropna()
    dates = pd.to_datetime(
        date_data['Date'].apply(lambda x: x + '0' if len(x) < 7 else x),
        format='%Y.%m'
    ) + MonthEnd(0)
    return dates

def process_main_data(file_path: str, sheet_name: str, col_names: list, nrows: int) -> pd.DataFrame:
    """Load and process the main data from the Excel file."""
    cape_data = pd.read_excel(
        file_path, 
        sheet_name=sheet_name, 
        skiprows=8, 
        header=None, 
        names=col_names, 
        nrows=nrows
    )
    cape_data.dropna(how='all', axis=1, inplace=True)
    cape_data.dropna(how='all', inplace=True)
    cape_data = cape_data.apply(pd.to_numeric, errors='coerce')
    return cape_data

def add_computed_columns(data: pd.DataFrame) -> pd.DataFrame:
    """Add computed columns to the DataFrame."""
    cpi_latest = data['Consumer Price Index CPI'].iloc[-1]
    data['Nominal Total Return Price'] = (
        data['Real Total Return Price'] * data['Consumer Price Index CPI'] / cpi_latest
    )
    data['Nominal Total Bond Returns'] = data['Monthly Total Bond Returns'].cumprod()
    return data

def trim_trailing_rows(data: pd.DataFrame, columns_to_check: list) -> pd.DataFrame:
    """Remove trailing rows with NaN values in specified columns."""
    last_valid_index = data[columns_to_check].last_valid_index()
    if last_valid_index is not None:
        data = data.loc[:last_valid_index]
    return data

def load_shiller_cape_data(file_url: str) -> pd.DataFrame:
    """
    Load and process Shiller CAPE data from a given URL.

    This function downloads an Excel file containing Shiller CAPE data,
    processes it, and returns a cleaned DataFrame with the data.
    """
    temp_file = '/tmp/shiller_cape.xls'
    sheet_name = 'Data'

    # Step 1: Download the file
    download_file(file_url, temp_file)

    # Step 2: Extract column names
    col_names = extract_column_names(temp_file, sheet_name)

    # Step 3: Parse and process dates
    dates = parse_dates(temp_file, sheet_name)

    # Step 4: Process main data
    cape_data = process_main_data(temp_file, sheet_name, col_names, len(dates))
    cape_data.index = dates  # Assign processed dates as index

    # Step 5: Add computed columns
    # cape_data = add_computed_columns(cape_data)

    # Step 6: Trim trailing rows
    # columns_to_check = [
    #     'Nominal Total Bond Returns',
    #     'Nominal Total Return Price',
    #     'Cyclically Adjusted Total Return Price Earnings Ratio TR P/E10 or TR CAPE'
    # ]
    # cape_data = trim_trailing_rows(cape_data, columns_to_check)

    return cape_data


In [ ]:
file_url = 'https://img1.wsimg.com/blobby/go/e5e77e0b-59d1-44d9-ab25-4763ac982e53/downloads/e1fcc664-eaa1-48ed-b682-88e0c33db496/ie_data.xls?ver=1733242673788'
cape_data = load_shiller_cape_data(file_url)

In [ ]:
cape_data.columns

In [ ]:
data_columns = {
    'S&P Comp. P': 'sp500Price'
    , 'Dividend D': 'sp500Dividend'
    , 'Earnings E': 'sp500Earnings'
    , 'Consumer Price Index CPI': 'cpi'
    , 'Long Interest Rate': 'gs10Rate'
    }